# LingoPrism – Fine-tune v5: model Open Images đang chạy, nhãn sạch

Lần này **không train từ đầu** và **không dùng nhãn tự gán** (hai lý do v1–v4 thua model gốc, xem
`AI/report/yolo/ghi_chu.md`). Notebook train tiếp đúng model server đang chạy (**yolov8m-oiv7**):

* giữ nguyên 601 lớp Open Images → mọi trọng số, kể cả đầu ra, được giữ lại (`Transferred 475/475`);
* đóng băng backbone, learning rate nhỏ → chỉnh nhẹ chứ không học lại;
* dữ liệu: Open Images V7 **train** (tập test của đồ án lấy từ validation → không lẫn), **chỉ nhãn do người vẽ**;
* **lớp yếu** (AP50 < 0,55 trên tập test 2 542 ảnh + các cặp hay nhầm trên ảnh thật như chanh/cam, dép/giày, violin/guitar)
  lấy nhiều ảnh; **mọi lớp còn lại** lấy ít ảnh để model không quên (replay).

**Cách chạy**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**, cho phép Google Drive.
3. Giữ tab mở. Bị ngắt thì *Run all* lại: dataset và checkpoint nằm trên Drive (`MyDrive/lingoprism_runs`), notebook làm tiếp.
4. Ô cuối tải về `oiv7m-ft-v5.zip` → gửi lại để chấm trên tập test 2 542 ảnh và quyết định có dùng hay không.

**Thời gian (Colab miễn phí, T4):** dựng dataset lần đầu ~1–1,5 giờ (tải nhãn + ảnh Open Images), sau đó giải nén
từ Drive vài phút; train tối đa `HOURS` giờ; đánh giá + export ~10 phút.

In [ ]:
# 1. GPU, mã nguồn, thư viện
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if not os.path.exists('/content/LingoPrism'):
    !git clone -q https://github.com/GPTii/LingoPrism.git /content/LingoPrism
else:
    !git -C /content/LingoPrism pull -q
%cd /content/LingoPrism
!pip install -q ultralytics fiftyone pyyaml onnx onnxslim onnxruntime
import torch
assert torch.cuda.is_available(), 'Chưa bật GPU: Runtime → Change runtime type → T4 GPU'

In [ ]:
# 2. Cấu hình – có thể chỉnh
import os
os.chdir('/content/LingoPrism')
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/lingoprism_runs'
os.makedirs(RUNS, exist_ok=True)

BASE = 'yolov8m-oiv7.pt'   # model server đang chạy (Ultralytics tự tải)
PER_CLASS = 300            # ảnh mỗi lớp Open Images của lớp yếu
REPLAY = 30                # ảnh mỗi lớp Open Images của các lớp còn lại (chống quên)
THRESHOLD = 0.55           # lớp yếu: AP50 trên tập test dưới ngưỡng này
EPOCHS = 30
HOURS = 3.5                # giới hạn cứng thời gian train
FREEZE = 10                # backbone YOLOv8 = 10 lớp đầu
LR0 = 0.002
BATCH = 16
NAME = 'oiv7m-ft-v5'
DS = f'/content/oiv7_ft_{PER_CLASS}_{REPLAY}'
TAR = f'{RUNS}/{os.path.basename(DS)}.tar'

In [ ]:
# 3. Dataset: Open Images V7 train, nhãn người vẽ, nhãn giữ nguyên 601 lớp của model gốc
#    Lần đầu tải + lọc (~1–1,5 giờ) rồi lưu .tar lên Drive; phiên sau giải nén lại (vài phút).
import json, shutil
ready = lambda: os.path.exists(f'{DS}/summary.json')
if not ready() and os.path.exists(TAR):
    print('Giải nén dataset từ Drive...')
    !tar -xf {TAR} -C /content
    if not ready():  # bản lưu từ một lần dựng bị lỗi giữa chừng: bỏ đi, dựng lại
        print('Bản lưu trên Drive chưa hoàn chỉnh → xoá và dựng lại')
        os.remove(TAR)
        shutil.rmtree(DS, ignore_errors=True)
if not ready():
    !python AI/scripts/build_oiv7_finetune.py --base {BASE} --out {DS} --per-class {PER_CLASS} --replay {REPLAY} --threshold {THRESHOLD}
    assert ready(), 'Dựng dataset bị lỗi – xem thông báo ngay phía trên rồi gửi lại ảnh chụp lỗi'
    print('Lưu dataset lên Drive...')
    !tar -cf {TAR} -C /content {os.path.basename(DS)}
!rm -f {DS}/labels/*.cache
s = json.load(open(f'{DS}/summary.json'))
print(f"{s['images']} ảnh – {len(s['weak'])} lớp yếu: {', '.join(s['weak'])}")
print(f"{len(s['replay'])} lớp ôn lại")
assert s['images'] > 100, 'Quá ít ảnh – xem các dòng [bỏ qua] khi dựng dataset'


In [ ]:
# 4. Fine-tune (tự train tiếp nếu phiên trước bị ngắt) + đo trước / sau trên ảnh val + export ONNX như server
!python AI/scripts/finetune_oiv7.py --data {DS}/data.yaml --base {BASE} --project {RUNS} --name {NAME} \
    --epochs {EPOCHS} --hours {HOURS} --freeze {FREEZE} --lr0 {LR0} --batch {BATCH}

In [ ]:
# 5. Kết quả trên ảnh val (601 lớp, cùng ảnh cho trước và sau)
from IPython.display import Image, display
run = f'{RUNS}/{NAME}'
r = json.load(open(f'{run}/report/finetune_summary.json'))
for k in ('val_before', 'val_after'):
    m = r[k]
    print(f"{'Trước' if k == 'val_before' else 'Sau  '}: P {m['precision']:.3f} | R {m['recall']:.3f} | mAP50 {m['map50']:.3f} | mAP50-95 {m['map50_95']:.3f}")
if os.path.exists(f'{run}/results.png'):
    display(Image(f'{run}/results.png', width=900))
print('Kết luận cuối cùng chấm trên tập test 2 542 ảnh của đồ án (máy phát triển, xem ô 6).')

In [ ]:
# 6. Nén & tải về → gửi file zip này lại
import glob, zipfile
from google.colab import files
out = f'/content/{NAME}.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{run}/export/*.onnx') + glob.glob(f'{run}/export/model_card.json'):
        z.write(f, os.path.basename(f))
    for f in ('results.csv', 'results.png', 'args.yaml'):
        if os.path.exists(f'{run}/{f}'):
            z.write(f'{run}/{f}', f)
    for f in glob.glob(f'{run}/report/*.json'):
        z.write(f, 'report/' + os.path.basename(f))
    z.write(f'{DS}/summary.json', 'summary.json')
print(out, round(os.path.getsize(out) / 1e6, 1), 'MB')
files.download(out)
# Trên máy phát triển (giải nén vào AI/runs/oiv7m-ft-v5/):
#   python AI/scripts/quantize_onnx.py AI/runs/oiv7m-ft-v5/yolov8m-oiv7-ft-lingoprism.onnx
#   python AI/scripts/eval_testset.py --model ft=AI/runs/oiv7m-ft-v5/yolov8m-oiv7-ft-lingoprism-int8.onnx
#   python AI/scripts/compare_finetune.py --name ft --run AI/runs/oiv7m-ft-v5